# Gen data

In [ ]:
import pandas as pd
from utils.dataset import gen_text_for_embedding
from models.specter import embed_texts
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/VRID_NLP/data_translated.csv")
df = df[df["Interdisciplinario"].notna()]
print(df.shape)

cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

#Lectura de codigos VRID Test
X = gen_text_for_embedding(df, cols, element_names=None, sep=" ")

# Parámetros para cargar modelo
BASE_MODEL = "allenai/specter2_base"
ADAPTER_NAME="allenai/specter2_classification"

X = embed_texts(
    df["text_for_embedding_translated"].fillna("").astype(str).tolist(),
    BASE_MODEL,
    ADAPTER_NAME
)

y = np.where(df["Transdisciplinario"]=="Transdisciplinario", 1, 0)
valores, conteos = np.unique(y, return_counts=True)
print(dict(zip(valores, conteos)))
print(X.shape, y.shape)

# Compute UMAP

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

def interactive_plot(X_umap):
    # Definir colores para cada clase (ejemplo: 0=gris, 1=verde, 2=rojo, etc.)
    colors = np.array(["blue", "green"])

    # Crear figura 3D
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    # Graficar puntos en 3D, asignando color manualmente
    scatter = ax.scatter(
        X_umap[:, 0], X_umap[:, 1], X_umap[:, 2],
        c=colors[y], s=30
    )

    # Leyenda manual
    unique_classes = np.unique(y)
    for cls in unique_classes:
        ax.scatter([], [], [], c=colors[cls], label=f"Clase {cls}")
    ax.legend(title="Clases")

    # Etiquetas de ejes
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_zlabel("UMAP-3")
    ax.view_init(elev=30, azim=45)

    plt.show()

In [ ]:
import umap

# Crear modelo UMAP para reducir a 3 dimensiones
umap_model = umap.UMAP(n_components=3, random_state=42)

# Ajustar y transformar
X_umap = umap_model.fit_transform(X)

#Dimensions
print(X_umap.shape)
#Plot data 3d
interactive_plot(X_umap)